In [ ]:
!pip install GEOparse gseapy lifelines pandas numpy matplotlib seaborn statsmodels scipy networkx -q

In [ ]:
import GEOparse
import pandas as pd
import numpy as np
import gseapy as gp
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import requests
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully")

In [ ]:
print("Downloading GSE39582... this may take a few minutes")
gse = GEOparse.get_GEO("GSE39582", destdir="./geo_data/", silent=True)
print("Download complete")
print(f"Title: {gse.metadata['title'][0]}")
print(f"Number of samples: {len(gse.gsms)}")

In [ ]:
expr_data = gse.pivot_samples('VALUE')
print(f"Expression matrix shape: {expr_data.shape}")

sample_metadata = gse.phenotype_data
print(f"Metadata samples: {len(sample_metadata)}")
print(f"Metadata columns: {len(sample_metadata.columns)}")

In [ ]:
kras_col = None
for col in sample_metadata.columns:
    if 'kras.mutation' in col.lower():
        kras_col = col
        break

if kras_col is None:
    for col in sample_metadata.columns:
        if 'kras' in col.lower():
            kras_col = col
            break

print(f"KRAS column found: {kras_col}")
print("\nValue distribution:")
print(sample_metadata[kras_col].value_counts(dropna=False).head(10))

In [ ]:
kras_values = sample_metadata[kras_col].astype(str).str.strip().str.upper()

sample_metadata['KRAS_group'] = kras_values.apply(
    lambda x: 'Mutant' if x == 'M' else ('WildType' if x == 'WT' else 'Unknown')
)

print("Group distribution:")
print(sample_metadata['KRAS_group'].value_counts())

valid = sample_metadata[sample_metadata['KRAS_group'].isin(['Mutant', 'WildType'])]

mutant_samples = valid[valid['KRAS_group'] == 'Mutant'].index.tolist()
wildtype_samples = valid[valid['KRAS_group'] == 'WildType'].index.tolist()

print(f"\nKRAS-Mutant samples: {len(mutant_samples)}")
print(f"KRAS-WildType samples: {len(wildtype_samples)}")

In [ ]:
mutant_valid = [s for s in mutant_samples if s in expr_data.columns]
wildtype_valid = [s for s in wildtype_samples if s in expr_data.columns]

print(f"Valid Mutant samples: {len(mutant_valid)}")
print(f"Valid WildType samples: {len(wildtype_valid)}")

all_valid = mutant_valid + wildtype_valid
expr_filtered = expr_data[all_valid].copy()

if expr_filtered.max().max() > 50:
    expr_filtered = np.log2(expr_filtered + 1)
    print("Data transformed to log2")

print(f"Final matrix shape: {expr_filtered.shape}")

In [ ]:
print("Calculating DEGs...")

results = []
for gene in expr_filtered.index:
    mutant_vals = expr_filtered.loc[gene, mutant_valid].dropna()
    wildtype_vals = expr_filtered.loc[gene, wildtype_valid].dropna()

    if len(mutant_vals) > 3 and len(wildtype_vals) > 3:
        t_stat, p_val = stats.ttest_ind(mutant_vals, wildtype_vals)
        log2fc = mutant_vals.mean() - wildtype_vals.mean()

        results.append({
            'gene': gene,
            'log2FC': log2fc,
            'pvalue': p_val,
            'mean_mutant': mutant_vals.mean(),
            'mean_wildtype': wildtype_vals.mean()
        })

deg_df = pd.DataFrame(results).dropna()
print(f"Total genes analyzed: {len(deg_df)}")

deg_df['adj_pvalue'] = multipletests(deg_df['pvalue'], method='fdr_bh')[1]

print(f"Total genes with adjusted p-value: {len(deg_df)}")

In [ ]:
gpl_id = list(gse.gpls.keys())[0]
gpl = gse.gpls[gpl_id]

gene_col = 'Gene Symbol'
probe_to_gene = dict(zip(gpl.table['ID'], gpl.table[gene_col]))

def clean_gene_symbol(symbol):
    if pd.isna(symbol) or symbol == '' or symbol == '---':
        return None
    if '///' in str(symbol):
        symbol = str(symbol).split('///')[0].strip()
    if str(symbol).startswith(('BC0', 'CTD-', 'KIAA', 'FLJ', 'LOC')):
        return None
    return symbol

deg_df['gene_symbol'] = deg_df['gene'].map(probe_to_gene)
deg_df['gene_symbol_clean'] = deg_df['gene_symbol'].apply(clean_gene_symbol)

deg_with_symbol = deg_df.dropna(subset=['gene_symbol_clean']).copy()

deg_with_symbol = deg_with_symbol.groupby('gene_symbol_clean').agg({
    'log2FC': 'mean',
    'pvalue': 'mean',
    'adj_pvalue': 'min',
    'mean_mutant': 'mean',
    'mean_wildtype': 'mean'
}).reset_index().rename(columns={'gene_symbol_clean': 'gene_symbol'})

print(f"Total unique genes: {len(deg_with_symbol)}")

In [ ]:
deg_significant = deg_with_symbol[
    (deg_with_symbol['adj_pvalue'] < 0.05) &
    (abs(deg_with_symbol['log2FC']) > 0.58)
].sort_values('adj_pvalue')

print(f"Significant DEGs: {len(deg_significant)}")
print("\nTop 20 genes:")
print(deg_significant.head(20)[['gene_symbol', 'log2FC', 'adj_pvalue']].to_string(index=False))

deg_significant.to_csv('KRAS_DEGs_final.csv', index=False)
print("\nSaved: KRAS_DEGs_final.csv")

In [ ]:
gene_list = deg_significant['gene_symbol'].tolist()
print(f"Genes for PPI: {len(gene_list)}")

string_api_url = "https://string-db.org/api"
output_format = "tsv"
method = "network"

params = {
    "identifiers": "%0d".join(gene_list),
    "species": 9606,
    "caller_identity": "colorectal_cancer_kras_project"
}

request_url = "/".join([string_api_url, output_format, method])
response = requests.post(request_url, data=params)

with open("string_network.tsv", "w") as f:
    f.write(response.text)

ppi_network = pd.read_csv("string_network.tsv", sep="\t")
print(f"Total interactions: {len(ppi_network)}")

ppi_network.to_csv("PPI_network.csv", index=False)
print("Saved: PPI_network.csv")

In [ ]:
G = nx.Graph()

for _, row in ppi_network.iterrows():
    G.add_edge(row['preferredName_A'], row['preferredName_B'], weight=row['score'])

print(f"Graph nodes: {G.number_of_nodes()}")
print(f"Graph edges: {G.number_of_edges()}")

degree_dict = dict(G.degree())
betweenness_dict = nx.betweenness_centrality(G)
closeness_dict = nx.closeness_centrality(G)

hub_df = pd.DataFrame({
    'gene': list(degree_dict.keys()),
    'degree': list(degree_dict.values()),
    'betweenness': [betweenness_dict.get(g, 0) for g in degree_dict.keys()],
    'closeness': [closeness_dict.get(g, 0) for g in degree_dict.keys()]
}).sort_values('degree', ascending=False)

print("\nHub genes based on degree:")
print(hub_df.to_string(index=False))

hub_df.to_csv('hub_genes.csv', index=False)
print("\nSaved: hub_genes.csv")

In [ ]:
hub_genes_list = hub_df[hub_df['degree'] >= 3]['gene'].tolist()
print(f"Hub genes for enrichment: {len(hub_genes_list)}")

enrichr_kegg = gp.enrichr(
    gene_list=hub_genes_list,
    gene_sets='KEGG_2021_Human',
    organism='human',
    outdir=None
)

print("\nKEGG results:")
print(enrichr_kegg.results[['Term', 'Overlap', 'P-value', 'Adjusted P-value']].head(10).to_string(index=False))

enrichr_go = gp.enrichr(
    gene_list=hub_genes_list,
    gene_sets='GO_Biological_Process_2021',
    organism='human',
    outdir=None
)

print("\nGO results:")
print(enrichr_go.results[['Term', 'Overlap', 'P-value', 'Adjusted P-value']].head(10).to_string(index=False))

enrichr_kegg.results.to_csv('KEGG_results.csv', index=False)
enrichr_go.results.to_csv('GO_results.csv', index=False)
print("\nSaved: KEGG_results.csv and GO_results.csv")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

csv_files = ['KRAS_DEGs_final.csv', 'PPI_network.csv', 'hub_genes.csv', 'KEGG_results.csv', 'GO_results.csv']

for f in csv_files:
    if os.path.exists(f):
        shutil.copy(f, '/content/drive/MyDrive/')
        print(f"Copied to Drive: {f}")
    else:
        print(f"Not found: {f}")